# 04 · Generación de datos ficticios con Faker y Spark

**Para qué:** necesitamos más volumen para los experimentos (5 y 10 millones de filas) y solo tenemos 1,048,575 reales.

**Cómo (sin pandas):**
1. Calculamos con Spark el *perfil* de los datos reales: qué valores aparecen en cada columna y con qué frecuencia.
2. Dividimos el trabajo en **bloques de 10,000 filas**. Spark reparte los bloques entre sus particiones y una función Python con **Faker** genera cada bloque eligiendo valores según ese perfil.
3. El resultado tiene **las mismas 22 columnas que el CSV original**. Por eso pasa por la misma limpieza Silver y también entra al ETL de SQL Server sin cambios.

Todo es Python y Spark: no se usa pandas ni ninguna otra librería de datos.

In [ ]:
%pip install -q faker

In [ ]:
%run ./00_utilidades

## 1. Perfil estadístico de los datos reales
Para cada columna contamos cuántas veces aparece cada valor. Después, Faker elegirá valores con esas probabilidades
(si el 70 % de los partos son de un tipo, ~70 % de los datos ficticios también lo serán).

In [ ]:
silver = leer_tabla(T_SILVER_1X)

def a_texto(v):
    """Convierte un valor a texto como aparece en el CSV original: 3390.0 -> '3390', 46.2 -> '46.2', nulo -> ''."""
    if v is None:
        return ""
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v)

def tabla_frecuencias(df, columnas):
    """Devuelve (valores, acumuladas): lo que necesita random.choices para elegir con probabilidades."""
    filas = df.groupBy(*columnas).count().collect()
    valores, acumuladas, total = [], [], 0
    for f in filas:
        v = tuple(a_texto(f[c]) for c in columnas)
        valores.append(v if len(columnas) > 1 else v[0])
        total += f["count"]
        acumuladas.append(total)
    return valores, acumuladas

PERFIL = {}
PERFIL[("anio", "mes")] = tabla_frecuencias(silver, ["anio", "mes"])      # año y mes se eligen juntos
for original in COLUMNAS_ORIGINALES:
    nombre = NOMBRE_SILVER[original]
    if nombre not in ("anio", "mes"):
        PERFIL[original] = tabla_frecuencias(silver, [nombre])
print("Columnas perfiladas:", len(PERFIL), "| valores totales:", sum(len(v[0]) for v in PERFIL.values()))

## 2. La función que genera los datos
Spark le entrega a la función **un número de bloque** y **cuántas filas** debe generar. La función devuelve esas filas ya con valores ficticios.

**¿Qué es la semilla?** Una computadora no genera números realmente al azar: sigue una fórmula que parte de un número inicial (la semilla).
Con la misma semilla sale siempre la misma secuencia, y así cualquiera puede repetir el experimento y obtener los mismos datos.

**Por qué `SEMILLA + número de bloque`:** si todos los bloques usaran la misma semilla, todos generarían filas idénticas.
Así cada bloque es distinto y, aun así, reproducible. El valor 42 no tiene un significado especial: es una convención, lo importante es fijarlo y documentarlo.

In [ ]:
from pyspark.sql.types import ArrayType

ESQUEMA_FILA = StructType([StructField("id", LongType(), True)] +
                          [StructField(c, StringType(), True) for c in COLUMNAS_ORIGINALES])
FILAS_POR_BLOQUE = 10_000

def generar_sintetico(filas, por_bloque=FILAS_POR_BLOQUE, particiones=32):
    perfil, semilla, columnas = PERFIL, SEMILLA, COLUMNAS_ORIGINALES

    def generar_bloque(bloque, n):
        """Genera n filas ficticias. Devuelve una lista de tuplas (id + las 22 columnas)."""
        from faker import Faker
        if not hasattr(generar_bloque, "_fake"):          # Faker se crea una sola vez por proceso
            generar_bloque._fake = Faker("es_ES")
        fake = generar_bloque._fake
        fake.seed_instance(semilla + bloque)              # semilla distinta por bloque
        azar = fake.random
        valores, acum = perfil[("anio", "mes")]
        pares = azar.choices(valores, cum_weights=acum, k=n)
        datos = {"FecNac_Año": [p[0] for p in pares], "FecNac_Mes": [p[1] for p in pares]}
        for c in columnas:
            if c not in datos:
                valores, acum = perfil[c]
                datos[c] = azar.choices(valores, cum_weights=acum, k=n)
        ids = range(bloque * por_bloque, bloque * por_bloque + n)
        return list(zip(ids, *[datos[c] for c in columnas]))

    generar_udf = F.udf(generar_bloque, ArrayType(ESQUEMA_FILA))
    n_bloques = -(-filas // por_bloque)                   # división hacia arriba
    return (spark.range(0, n_bloques, numPartitions=min(particiones, n_bloques))
            .select(F.explode(generar_udf(
                F.col("id").cast("int"),
                F.least(F.lit(por_bloque), F.lit(filas) - F.col("id") * por_bloque).cast("int"))).alias("fila"))
            .select("fila.*"))

## 3. Probar con pocas filas
Siempre se prueba primero con 100,000 filas. Si funciona, se genera el volumen completo.

In [ ]:
prueba = generar_sintetico(100_000)
display(prueba.limit(5))
print("Filas:", prueba.count())
print("Filas únicas (sin contar id):", prueba.drop("id").dropDuplicates().count())

## 4. Generar los 9 millones y guardarlos
Es el paso más lento. Con 16 particiones trabajando a la vez puede tardar varios minutos.

In [ ]:
t0 = time.time()
guardar_tabla(generar_sintetico(FILAS_SINT_10X), T_SINTETICO)
sint = leer_tabla(T_SINTETICO)
print(f"Sintético guardado: {sint.count():,} filas en {time.time() - t0:.0f} s")

## 5. ¿Se parecen a los reales?
Pasamos los datos ficticios por la **misma limpieza** y comparamos los indicadores.

In [ ]:
kpi_real = calcular_kpis(leer_tabla(T_SILVER_1X)).withColumn("origen", F.lit("real"))
kpi_sint = calcular_kpis(limpiar_a_silver(sint.drop("id"))).withColumn("origen", F.lit("sintetico"))
display(kpi_real.unionByName(kpi_sint))

## 6. Construir las escalas 5x y 10x en Silver
- **5x** = reales + las primeras 4,000,000 filas ficticias (`id < 4,000,000`)
- **10x** = reales + las 9,000,000 filas ficticias

La escala 5x contiene exactamente las mismas filas ficticias que se exportan a SQL Server, para que la comparación sea justa.

In [ ]:
silver_1x = leer_tabla(T_SILVER_1X)
for filas_sint, tabla in [(FILAS_SINT_5X, T_SILVER_5X), (FILAS_SINT_10X, T_SILVER_10X)]:
    t0 = time.time()
    sint_limpio = limpiar_a_silver(leer_tabla(T_SINTETICO).where(F.col("id") < filas_sint).drop("id"))
    guardar_tabla(silver_1x.unionByName(sint_limpio), tabla, particion="anio")
    print(f"{tabla}: {leer_tabla(tabla).count():,} filas en {time.time() - t0:.0f} s")

## 7. Exportar CSV para SQL Server
Se crean archivos con **las 22 columnas originales** y el mismo separador `;`, así el script `EXTRACCION DE DATOS.sql` los carga sin cambios.
Cada archivo contiene solo las filas ficticias (las reales ya están en tu base de datos).

In [ ]:
def exportar_csv(df, nombre_archivo):
    tmp = f"{RUTA_EXPORT}/_tmp_{nombre_archivo}"
    (df.coalesce(1).write.mode("overwrite")
       .option("header", True).option("sep", ";").option("encoding", "UTF-8")
       .option("emptyValue", "").option("nullValue", "").csv(tmp))
    parte = [f.path for f in dbutils.fs.ls(tmp) if f.name.startswith("part-") and f.name.endswith(".csv")][0]
    dbutils.fs.mv(parte, f"{RUTA_EXPORT}/{nombre_archivo}.csv", True)
    dbutils.fs.rm(tmp, True)
    print("Exportado:", f"{RUTA_EXPORT}/{nombre_archivo}.csv")

sint = leer_tabla(T_SINTETICO)
for filas, nombre in [(1_000_000, "sintetico_1M"), (4_000_000, "sintetico_4M")]:
    exportar_csv(sint.where(F.col("id") < filas).drop("id"), nombre)
display(dbutils.fs.ls(RUTA_EXPORT))